In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
train_identity=pd.read_csv('ieee-fraud-detection/train_identity.csv')
train_transaction=pd.read_csv('ieee-fraud-detection/train_transaction.csv')

In [4]:
# Columns we initially care about
transaction_cols = [
    "TransactionID",
    "isFraud",
    "TransactionDT",
    "TransactionAmt",
    "ProductCD",

    # Payment/card information
    "card1", "card2", "card3", "card4", "card5", "card6",

    # Address information
    "addr1", "addr2",

    # Email information
    "P_emaildomain",
    "R_emaildomain",
]

identity_cols = [
    "TransactionID",
    "DeviceType",
    "DeviceInfo",
] + [f"id_{i:02d}" for i in range(1, 39)]

tx = train_transaction[transaction_cols].copy()

identity = train_identity[identity_cols].copy()

df = tx.merge(
    identity,
    on="TransactionID",
    how="left"
)

In [5]:
device_network = (
    df.dropna(subset=["DeviceInfo", "card1"])
      .groupby("DeviceInfo")
      .agg(
          transactions=("TransactionID", "count"),
          unique_cards=("card1", "nunique"),
          frauds=("isFraud", "sum"),
          fraud_rate=("isFraud", "mean")
      )
)

In [6]:
import networkx as nx

# Use only specific/repeated DeviceInfo signatures.
# Exclude extremely generic values for now.
device_counts = (
    df["DeviceInfo"]
    .value_counts(dropna=True)
)

# Initial heuristic:
# DeviceInfo must appear in <= 1000 transactions
# and at least 2 different cards.
candidate_devices = set(
    device_network[
        (device_network["transactions"] <= 1000) &
        (device_network["unique_cards"] >= 2)
    ].index
)

graph_df = df[
    df["card1"].notna() &
    df["DeviceInfo"].isin(candidate_devices)
].copy()

print("Transactions used:", len(graph_df))
print("Cards:", graph_df["card1"].nunique())
print("Device signatures:", graph_df["DeviceInfo"].nunique())

Transactions used: 28468
Cards: 2672
Device signatures: 1232


In [7]:
edge_stats = (
    graph_df
    .groupby(["card1", "DeviceInfo"])
    .agg(
        transactions=("TransactionID", "count"),
        frauds=("isFraud", "sum"),
        fraud_rate=("isFraud", "mean")
    )
    .reset_index()
)

print(edge_stats.head(20).to_string(index=False))

 card1                         DeviceInfo  transactions  frauds  fraud_rate
  1000           F3213 Build/36.0.A.2.146             1       0         0.0
  1011              SM-G950U Build/NRD90M             1       0         0.0
  1012                            rv:48.0             1       0         0.0
  1014            Ilium X220 Build/MRA58K             1       0         0.0
  1015       Moto E (4) Build/NMA26.42-69             1       0         0.0
  1015                               PH-1             2       0         0.0
  1015                              Pixel             1       0         0.0
  1015                            SAMSUNG             1       0         0.0
  1018              SM-J200M Build/LMY47X             1       0         0.0
  1023              SM-G950U Build/NRD90M             1       0         0.0
  1027              SM-G920F Build/NRD90M             1       0         0.0
  1027              SM-G920P Build/NRD90M             1       0         0.0
  1030      

In [8]:
G = nx.Graph()

for _, row in edge_stats.iterrows():

    card_node = f"card_{int(row['card1'])}"
    device_node = f"device_{row['DeviceInfo']}"

    G.add_node(
        card_node,
        node_type="card"
    )

    G.add_node(
        device_node,
        node_type="device"
    )

    G.add_edge(
        card_node,
        device_node,
        transactions=int(row["transactions"]),
        frauds=int(row["frauds"]),
        fraud_rate=float(row["fraud_rate"])
    )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 3904
Edges: 12785


In [9]:
target = "SM-A300H Build/LRX22G"
target_node = f"device_{target}"

neighbors = list(G.neighbors(target_node))

print("Cards connected to target:", len(neighbors))

for card in neighbors:
    edge = G[target_node][card]

    print(
        card,
        "transactions =", edge["transactions"],
        "frauds =", edge["frauds"],
        "fraud_rate =", round(edge["fraud_rate"], 3)
    )

Cards connected to target: 52
card_1976 transactions = 11 frauds = 11 fraud_rate = 1.0
card_2256 transactions = 20 frauds = 20 fraud_rate = 1.0
card_2650 transactions = 2 frauds = 0 fraud_rate = 0.0
card_2801 transactions = 4 frauds = 4 fraud_rate = 1.0
card_3154 transactions = 5 frauds = 2 fraud_rate = 0.4
card_3867 transactions = 4 frauds = 4 fraud_rate = 1.0
card_3887 transactions = 2 frauds = 2 fraud_rate = 1.0
card_3901 transactions = 1 frauds = 1 fraud_rate = 1.0
card_4461 transactions = 3 frauds = 1 fraud_rate = 0.333
card_4504 transactions = 7 frauds = 7 fraud_rate = 1.0
card_5009 transactions = 1 frauds = 0 fraud_rate = 0.0
card_5365 transactions = 1 frauds = 1 fraud_rate = 1.0
card_5535 transactions = 1 frauds = 0 fraud_rate = 0.0
card_5595 transactions = 1 frauds = 0 fraud_rate = 0.0
card_5629 transactions = 1 frauds = 1 fraud_rate = 1.0
card_5812 transactions = 12 frauds = 11 fraud_rate = 0.917
card_5943 transactions = 10 frauds = 10 fraud_rate = 1.0
card_7099 transactions 

In [10]:
card_device_counts = {}

for card in neighbors:
    device_neighbors = [
        n for n in G.neighbors(card)
        if G.nodes[n]["node_type"] == "device"
    ]

    card_device_counts[card] = len(device_neighbors)

for card, count in sorted(
    card_device_counts.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]:

    print(card, "→", count, "device signatures")

card_15885 → 552 device signatures
card_3154 → 383 device signatures
card_9633 → 374 device signatures
card_5812 → 289 device signatures
card_4461 → 276 device signatures
card_16136 → 248 device signatures
card_13832 → 247 device signatures
card_9026 → 187 device signatures
card_11201 → 185 device signatures
card_2256 → 165 device signatures
card_10568 → 156 device signatures
card_16062 → 146 device signatures
card_8755 → 141 device signatures
card_9917 → 128 device signatures
card_14276 → 123 device signatures
card_2650 → 107 device signatures
card_10876 → 105 device signatures
card_1976 → 102 device signatures
card_4504 → 94 device signatures
card_5365 → 76 device signatures


In [11]:
from itertools import combinations

card_pairs = []

for card1, card2 in combinations(neighbors, 2):

    devices1 = set(G.neighbors(card1))
    devices2 = set(G.neighbors(card2))

    shared_devices = devices1.intersection(devices2)

    if shared_devices:
        card_pairs.append({
            "card1": card1,
            "card2": card2,
            "shared_devices": len(shared_devices)
        })

card_pairs = sorted(
    card_pairs,
    key=lambda x: x["shared_devices"],
    reverse=True
)

for x in card_pairs[:30]:
    print(x)

{'card1': 'card_9633', 'card2': 'card_15885', 'shared_devices': 307}
{'card1': 'card_3154', 'card2': 'card_15885', 'shared_devices': 298}
{'card1': 'card_3154', 'card2': 'card_9633', 'shared_devices': 242}
{'card1': 'card_5812', 'card2': 'card_15885', 'shared_devices': 229}
{'card1': 'card_4461', 'card2': 'card_15885', 'shared_devices': 222}
{'card1': 'card_13832', 'card2': 'card_15885', 'shared_devices': 212}
{'card1': 'card_3154', 'card2': 'card_4461', 'shared_devices': 197}
{'card1': 'card_5812', 'card2': 'card_9633', 'shared_devices': 196}
{'card1': 'card_15885', 'card2': 'card_16136', 'shared_devices': 195}
{'card1': 'card_3154', 'card2': 'card_5812', 'shared_devices': 192}
{'card1': 'card_4461', 'card2': 'card_9633', 'shared_devices': 183}
{'card1': 'card_3154', 'card2': 'card_16136', 'shared_devices': 180}
{'card1': 'card_3154', 'card2': 'card_13832', 'shared_devices': 179}
{'card1': 'card_9633', 'card2': 'card_13832', 'shared_devices': 176}
{'card1': 'card_9026', 'card2': 'card

In [12]:
identity_candidate_cols = [
    "DeviceInfo",
    "DeviceType"
] + [f"id_{i:02d}" for i in range(12, 39)]

identity_stats = []

for col in identity_candidate_cols:

    temp = df[[col, "card1", "isFraud"]].dropna(subset=[col])

    grouped = (
        temp.groupby(col)
        .agg(
            transactions=("card1", "size"),
            unique_cards=("card1", "nunique"),
            frauds=("isFraud", "sum")
        )
    )

    identity_stats.append({
        "feature": col,
        "unique_values": temp[col].nunique(),
        "transactions": len(temp),
        "median_transactions_per_value": grouped["transactions"].median(),
        "max_transactions_per_value": grouped["transactions"].max(),
        "median_cards_per_value": grouped["unique_cards"].median(),
        "max_cards_per_value": grouped["unique_cards"].max()
    })

identity_stats = pd.DataFrame(identity_stats)

print(identity_stats.to_string(index=False))

   feature  unique_values  transactions  median_transactions_per_value  max_transactions_per_value  median_cards_per_value  max_cards_per_value
DeviceInfo           1786        118666                            4.0                       47722                     3.0                 4846
DeviceType              2        140810                        70405.0                       85165                  5845.0                 6709
     id_12              2        144233                        72116.5                      123025                  5514.5                 8282
     id_13             54        127320                           24.0                       58099                    15.5                 5312
     id_14             25         80044                           62.0                       44121                    31.0                 4229
     id_15              3        140985                        61612.0                       67728                  5690.0              

In [13]:
import pandas as pd
import numpy as np

signature_candidates = {
    "DeviceInfo": [
        "DeviceInfo"
    ],

    "DeviceInfo + id30": [
        "DeviceInfo", "id_30"
    ],

    "DeviceInfo + id30 + id31": [
        "DeviceInfo", "id_30", "id_31"
    ],

    "DeviceInfo + id30 + id31 + id33": [
        "DeviceInfo", "id_30", "id_31", "id_33"
    ],

    "DeviceInfo + id30 + id31 + id33 + id17": [
        "DeviceInfo", "id_30", "id_31", "id_33", "id_17"
    ]
}

results = []

for name, cols in signature_candidates.items():

    temp = df[["card1"] + cols].copy()

    # Only use rows where every component of the
    # candidate signature is actually present.
    temp = temp.dropna(subset=["card1"] + cols)

    # Create composite signature
    temp["signature"] = (
        temp[cols]
        .astype(str)
        .agg("|".join, axis=1)
    )

    grouped = (
        temp.groupby("signature")
        .agg(
            transactions=("card1", "size"),
            unique_cards=("card1", "nunique")
        )
    )

    results.append({
        "signature": name,
        "rows": len(temp),
        "unique_signatures": len(grouped),
        "signatures_with_2plus_cards": (
            (grouped["unique_cards"] >= 2).sum()
        ),
        "signatures_with_5plus_cards": (
            (grouped["unique_cards"] >= 5).sum()
        ),
        "median_cards_per_signature": (
            grouped["unique_cards"].median()
        ),
        "max_cards_per_signature": (
            grouped["unique_cards"].max()
        ),
        "median_transactions": (
            grouped["transactions"].median()
        ),
        "max_transactions": (
            grouped["transactions"].max()
        )
    })

signature_results = pd.DataFrame(results)

print(signature_results.to_string(index=False))

                             signature   rows  unique_signatures  signatures_with_2plus_cards  signatures_with_5plus_cards  median_cards_per_signature  max_cards_per_signature  median_transactions  max_transactions
                            DeviceInfo 118666               1786                         1237                          659                         3.0                     4846                  4.0             47722
                     DeviceInfo + id30  75539                881                          561                          295                         2.0                     2989                  3.0             19015
              DeviceInfo + id30 + id31  75306               2068                         1036                          487                         2.0                     1487                  2.0              5822
       DeviceInfo + id30 + id31 + id33  71120               4855                         2516                         1242                  

In [14]:
identity_cols = [
    f"id_{i:02d}" for i in range(1, 39)
]

results = []

for col in identity_cols:

    temp = df[["card1", col]].dropna()

    grouped = (
        temp.groupby(col)["card1"]
        .agg(
            transactions="size",
            unique_cards="nunique"
        )
    )

    results.append({
        "feature": col,
        "rows": len(temp),
        "unique_values": temp[col].nunique(),

        # How many values connect multiple cards?
        "values_2plus_cards": (
            grouped["unique_cards"] >= 2
        ).sum(),

        "values_5plus_cards": (
            grouped["unique_cards"] >= 5
        ).sum(),

        "median_cards_per_value":
            grouped["unique_cards"].median(),

        "p90_cards_per_value":
            grouped["unique_cards"].quantile(0.90),

        "max_cards_per_value":
            grouped["unique_cards"].max()
    })

identity_screen = (
    pd.DataFrame(results)
    .sort_values(
        ["values_5plus_cards", "unique_values"],
        ascending=False
    )
)

print(identity_screen.to_string(index=False))

feature   rows  unique_values  values_2plus_cards  values_5plus_cards  median_cards_per_value  p90_cards_per_value  max_cards_per_value
  id_19 139318            522                 469                 368                     9.0                 86.9                 2898
  id_20 139261            394                 270                 161                     3.0                210.5                 1616
  id_11 140978            365                 265                 157                     3.0                 30.6                 8165
  id_31 140282            130                 114                 106                    40.0                802.6                 3137
  id_33  73289            260                 154                 100                     2.0                116.0                 3058
  id_06 136865            101                  98                  91                    59.0                487.0                 6909
  id_08   5155             94                  8

In [15]:
import pandas as pd
import numpy as np

# ============================================================
# TEST 6: TEMPORAL STABILITY OF CANDIDATE GRAPH ENTITIES
# ============================================================

# ------------------------------------------------------------
# 1. Build the working dataframe
# ------------------------------------------------------------

# If you already have a merged dataframe containing identity
# columns, use that dataframe instead.
#
# Otherwise:
df = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print("Dataset:", df.shape)


# ------------------------------------------------------------
# 2. Candidate graph entities
# ------------------------------------------------------------

candidate_features = [
    "DeviceInfo",
    "id_19",
    "id_20",
    "id_31",
    "id_06",
    "id_30",
    "id_33",
    "id_05",
    "id_08",
    "id_17",
    "id_21",
    "addr1",
    "addr2",
    "card2",
    "card3",
    "card5"
]

candidate_features = [
    c for c in candidate_features
    if c in df.columns
]

print("\nFeatures being tested:")
print(candidate_features)


# ------------------------------------------------------------
# 3. Create chronological windows
# ------------------------------------------------------------

df = df.sort_values("TransactionDT").reset_index(drop=True)

df["time_window"] = pd.qcut(
    df["TransactionDT"],
    q=5,
    labels=False,
    duplicates="drop"
)

print("\nTransactions per window:")
print(df["time_window"].value_counts().sort_index())


# ------------------------------------------------------------
# 4. Measure temporal behaviour
# ------------------------------------------------------------

results = []

for feature in candidate_features:

    temp = df[
        ["TransactionID",
         "TransactionDT",
         "isFraud",
         "card1",
         feature,
         "time_window"]
    ].copy()

    temp = temp.dropna(subset=[feature])

    # Convert values to string so mixed datatypes don't cause
    # grouping problems.
    temp[feature] = temp[feature].astype(str)

    for window in sorted(temp["time_window"].unique()):

        w = temp[temp["time_window"] == window]

        entity_stats = (
            w.groupby(feature)
             .agg(
                 transactions=("TransactionID", "size"),
                 unique_cards=("card1", "nunique"),
                 fraud_rate=("isFraud", "mean")
             )
        )

        results.append({
            "feature": feature,
            "window": int(window),

            "entities": len(entity_stats),

            "multi_card_entities":
                (entity_stats["unique_cards"] >= 2).sum(),

            "5plus_card_entities":
                (entity_stats["unique_cards"] >= 5).sum(),

            "10plus_card_entities":
                (entity_stats["unique_cards"] >= 10).sum(),

            "median_cards":
                entity_stats["unique_cards"].median(),

            "p90_cards":
                entity_stats["unique_cards"].quantile(0.90),

            "max_cards":
                entity_stats["unique_cards"].max(),

            "fraud_rate":
                w["isFraud"].mean()
        })


temporal_results = pd.DataFrame(results)

print("\n================ TEMPORAL RESULTS ================\n")

print(
    temporal_results.to_string(index=False)
)

Dataset: (590540, 434)

Features being tested:
['DeviceInfo', 'id_19', 'id_20', 'id_31', 'id_06', 'id_30', 'id_33', 'id_05', 'id_08', 'id_17', 'id_21', 'addr1', 'addr2', 'card2', 'card3', 'card5']

Transactions per window:
time_window
0    118108
1    118108
2    118108
3    118108
4    118108
Name: count, dtype: int64


C:\Users\Anirban Banerjee\AppData\Local\Temp\ipykernel_35640\229016336.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["time_window"] = pd.qcut(



================ TEMPORAL RESULTS ================

   feature  window  entities  multi_card_entities  5plus_card_entities  10plus_card_entities  median_cards  p90_cards  max_cards  fraud_rate
DeviceInfo       0       975                  598                  250                   100           2.0       10.0       3152    0.024808
DeviceInfo       1       858                  481                  184                    72           2.0        8.3       1515    0.116546
DeviceInfo       2       835                  436                  175                    70           2.0        8.0       1315    0.106270
DeviceInfo       3       908                  498                  216                    97           2.0       10.0       1360    0.103679
DeviceInfo       4       848                  458                  179                    69           2.0        8.3       1425    0.088813
     id_19       0       456                  385                  236                   153         

In [16]:
# ============================================================
# TEMPORAL SUMMARY
# ============================================================

temporal_summary = (
    temporal_results
    .groupby("feature")
    .agg(
        median_multi_card_entities=(
            "multi_card_entities", "median"
        ),

        median_5plus_entities=(
            "5plus_card_entities", "median"
        ),

        median_10plus_entities=(
            "10plus_card_entities", "median"
        ),

        median_cards_per_entity=(
            "median_cards", "median"
        ),

        median_p90_cards=(
            "p90_cards", "median"
        ),

        max_cards_observed=(
            "max_cards", "max"
        ),

        min_window_multi_card=(
            "multi_card_entities", "min"
        ),

        max_window_multi_card=(
            "multi_card_entities", "max"
        )
    )
    .sort_values(
        "median_multi_card_entities",
        ascending=False
    )
)

print(temporal_summary)

            median_multi_card_entities  median_5plus_entities  \
feature                                                         
DeviceInfo                       481.0                  184.0   
card2                            380.0                  148.0   
id_19                            248.0                  128.0   
id_20                            145.0                   97.0   
addr1                             84.0                   65.0   
id_06                             80.0                   63.0   
id_31                             77.0                   68.0   
id_33                             76.0                   54.0   
id_05                             67.0                   52.0   
id_30                             66.0                   58.0   
id_08                             61.0                   43.0   
card3                             53.0                   34.0   
card5                             50.0                   39.0   
id_21                    

In [17]:
# ============================================================
# 1. TEMPORAL TRAIN / TEST SPLIT
# ============================================================

df = df.sort_values("TransactionDT").reset_index(drop=True)

split_idx = int(len(df) * 0.80)

train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()

print("Train:", train_df.shape)
print("Test :", test_df.shape)

print(
    "Train fraud rate:",
    train_df["isFraud"].mean()
)

print(
    "Test fraud rate:",
    test_df["isFraud"].mean()
)

Train: (472432, 435)
Test : (118108, 435)
Train fraud rate: 0.03513521522674162
Test fraud rate: 0.034409184813899145


In [18]:
# ============================================================
# GRAPH ABLATION CONFIGURATIONS
# ============================================================

graph_configs = {

    "G1_Device": [
        "DeviceInfo"
    ],

    "G2_Device_addr": [
        "DeviceInfo",
        "addr1"
    ],

    "G3_Device_addr_id19": [
        "DeviceInfo",
        "addr1",
        "id_19"
    ],

    "G4_plus_id20": [
        "DeviceInfo",
        "addr1",
        "id_19",
        "id_20"
    ],

    "G5_plus_id31": [
        "DeviceInfo",
        "addr1",
        "id_19",
        "id_20",
        "id_31"
    ],

    "G6_extended": [
        "DeviceInfo",
        "addr1",
        "id_19",
        "id_20",
        "id_31",
        "id_06",
        "id_30",
        "id_33",
        "id_05",
        "id_08",
        "id_17",
        "id_21",
        "addr2",
        "card2",
        "card3",
        "card5"
    ]
}

In [19]:
# ============================================================
# GRAPH ABLATION BASELINE
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)


def build_entity_risk(
    train_data,
    features,
    min_entity_transactions=3
):
    """
    Learn historical fraud risk for every entity.

    IMPORTANT:
    Only training data is used here.
    """

    risk_tables = {}

    for feature in features:

        if feature not in train_data.columns:
            continue

        temp = train_data[
            [feature, "isFraud"]
        ].dropna().copy()

        temp[feature] = temp[feature].astype(str)

        stats = (
            temp.groupby(feature)
            .agg(
                transactions=("isFraud", "size"),
                frauds=("isFraud", "sum")
            )
        )

        stats = stats[
            stats["transactions"] >= min_entity_transactions
        ].copy()

        # Smoothed fraud rate
        alpha = 2

        stats["risk"] = (
            stats["frauds"] + alpha * train_data["isFraud"].mean()
        ) / (
            stats["transactions"] + alpha
        )

        risk_tables[feature] = stats["risk"]

    return risk_tables


def score_transactions(
    test_data,
    risk_tables,
    features
):
    """
    For every transaction, collect risk from all connected
    entities and aggregate them.
    """

    scores = pd.DataFrame(
        index=test_data.index
    )

    for feature in features:

        if feature not in test_data.columns:
            continue

        mapping = risk_tables.get(feature)

        if mapping is None:
            continue

        values = (
            test_data[feature]
            .astype(str)
            .map(mapping)
        )

        scores[feature] = values

    # Number of graph entities that matched
    scores["entity_count"] = scores.notna().sum(axis=1)

    # Missing entity risk = global train fraud rate
    global_risk = train_df["isFraud"].mean()

    scores = scores.fillna(global_risk)

    # --------------------------------------------------------
    # Main graph risk score
    # --------------------------------------------------------

    scores["graph_risk"] = scores[features].mean(axis=1)

    return scores

In [20]:
# ============================================================
# RUN ABLATION
# ============================================================

ablation_results = []

for name, features in graph_configs.items():

    print("\nRunning:", name)
    print("Features:", features)

    available_features = [
        f for f in features
        if f in train_df.columns
    ]

    # --------------------------------------------
    # Learn graph/entity risk ONLY from TRAIN
    # --------------------------------------------

    risk_tables = build_entity_risk(
        train_df,
        available_features,
        min_entity_transactions=3
    )

    # --------------------------------------------
    # Score TEST transactions
    # --------------------------------------------

    scores = score_transactions(
        test_df,
        risk_tables,
        available_features
    )

    y_true = test_df["isFraud"]

    y_score = scores["graph_risk"]

    # --------------------------------------------
    # Metrics
    # --------------------------------------------

    auc = roc_auc_score(
        y_true,
        y_score
    )

    ap = average_precision_score(
        y_true,
        y_score
    )

    # Top 1% precision
    n_top = max(
        1,
        int(len(test_df) * 0.01)
    )

    top_idx = (
        y_score
        .nlargest(n_top)
        .index
    )

    precision_top1 = (
        y_true.loc[top_idx].mean()
    )

    lift_top1 = (
        precision_top1 /
        y_true.mean()
    )

    ablation_results.append({
        "graph": name,
        "num_features": len(available_features),
        "features": ", ".join(available_features),

        "ROC_AUC": auc,
        "PR_AUC": ap,

        "Top1_precision": precision_top1,
        "Top1_lift": lift_top1
    })


ablation_df = pd.DataFrame(
    ablation_results
).sort_values(
    "PR_AUC",
    ascending=False
)

print("\n================ ABLATION RESULTS ================\n")

print(
    ablation_df.to_string(index=False)
)


Running: G1_Device
Features: ['DeviceInfo']

Running: G2_Device_addr
Features: ['DeviceInfo', 'addr1']

Running: G3_Device_addr_id19
Features: ['DeviceInfo', 'addr1', 'id_19']

Running: G4_plus_id20
Features: ['DeviceInfo', 'addr1', 'id_19', 'id_20']

Running: G5_plus_id31
Features: ['DeviceInfo', 'addr1', 'id_19', 'id_20', 'id_31']

Running: G6_extended
Features: ['DeviceInfo', 'addr1', 'id_19', 'id_20', 'id_31', 'id_06', 'id_30', 'id_33', 'id_05', 'id_08', 'id_17', 'id_21', 'addr2', 'card2', 'card3', 'card5']

================ ABLATION RESULTS ================

              graph  num_features                                                                                                            features  ROC_AUC   PR_AUC  Top1_precision  Top1_lift
        G6_extended            16 DeviceInfo, addr1, id_19, id_20, id_31, id_06, id_30, id_33, id_05, id_08, id_17, id_21, addr2, card2, card3, card5 0.759439 0.153502        0.305673   8.883476
       G4_plus_id20             4      

## Feature Selection:
here in this module we want to figure out which features among the whole dataset of IEEE-CIS actually matters for our abuse ring detector model. 
We will implement greedy algorithms minimise PR-AUC

In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score


candidate_features = [
    "DeviceInfo",
    "addr1",
    "id_19",
    "id_20",
    "id_31",
    "id_06",
    "id_30",
    "id_33",
    "id_05",
    "id_08",
    "id_17",
    "id_21",
    "addr2",
    "card2",
    "card3",
    "card5"
]


# ---------------------------------------------------------
# 1. Fixed train / validation split
# ---------------------------------------------------------

train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["isFraud"],
    random_state=42
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

print(
    "Train fraud rate:",
    train_df["isFraud"].mean()
)

print(
    "Validation fraud rate:",
    val_df["isFraud"].mean()
)

Train: (472432, 435)
Validation: (118108, 435)
Train fraud rate: 0.03498916246147594
Validation fraud rate: 0.0349933958749619


In [22]:
def fit_entity_statistics(train_df, features):

    stats = {}

    for feature in features:

        tmp = train_df[
            ["card1", feature, "isFraud"]
        ].dropna(subset=[feature])

        stats[feature] = {
            "card_count":
                tmp.groupby(feature)["card1"].nunique(),

            "tx_count":
                tmp.groupby(feature).size(),

            "fraud_rate":
                tmp.groupby(feature)["isFraud"].mean()
        }

    return stats

In [23]:
def transform_entity_features(data, stats):

    result = pd.DataFrame(index=data.index)

    for feature, feature_stats in stats.items():

        result[f"{feature}_card_count"] = (
            data[feature]
            .map(feature_stats["card_count"])
            .fillna(0)
        )

        result[f"{feature}_tx_count"] = (
            data[feature]
            .map(feature_stats["tx_count"])
            .fillna(0)
        )

        result[f"{feature}_fraud_rate"] = (
            data[feature]
            .map(feature_stats["fraud_rate"])
            .fillna(0)
        )

    return result

In [24]:
stats = fit_entity_statistics(
    train_df,
    candidate_features
)

X_train = transform_entity_features(
    train_df,
    stats
)

X_val = transform_entity_features(
    val_df,
    stats
)

y_train = train_df["isFraud"].values
y_val = val_df["isFraud"].values

print(X_train.shape)
print(X_val.shape)

(472432, 48)
(118108, 48)


In [27]:
# ============================================================
# PARAMETERS
# ============================================================

# Smoothing prevents:
# 1 transaction + 1 fraud = 100% risk

SMOOTHING = 20

GLOBAL_FRAUD_RATE = train_df["isFraud"].mean()


# ============================================================
# TOP-K METRICS
# ============================================================

def top_k_metrics(y_true, scores, fraction=0.01):

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    n = len(y_true)

    k = max(1, int(n * fraction))

    # Highest-risk transactions
    order = np.argsort(-scores)

    top_labels = y_true[order[:k]]

    precision = top_labels.mean()

    baseline = y_true.mean()

    lift = precision / baseline if baseline > 0 else np.nan

    return precision, lift


# ============================================================
# TEST EACH ENTITY INDIVIDUALLY
# ============================================================

results = []


for feature in candidate_features:

    print(f"Testing: {feature}")

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    tmp = train_df[
        ["card1", feature, "isFraud"]
    ].dropna(subset=[feature])

    # Entity transaction count
    tx_count = (
        tmp.groupby(feature)
        .size()
    )

    # Entity fraud count
    fraud_count = (
        tmp.groupby(feature)["isFraud"]
        .sum()
    )

    # --------------------------------------------------------
    # SMOOTHED ENTITY FRAUD RATE
    # --------------------------------------------------------

    # Bayesian-style smoothing:
    #
    # (frauds + alpha * global_rate)
    # --------------------------------
    # (transactions + alpha)

    entity_risk = (
        fraud_count.add(SMOOTHING * GLOBAL_FRAUD_RATE, fill_value=0)
        /
        tx_count.add(SMOOTHING, fill_value=0)
    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    scores = (
        val_df[feature]
        .map(entity_risk)
        .fillna(GLOBAL_FRAUD_RATE)
        .values
    )

    y_val = val_df["isFraud"].values

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    pr_auc = average_precision_score(
        y_val,
        scores
    )

    roc_auc = roc_auc_score(
        y_val,
        scores
    )

    precision_1, lift_1 = top_k_metrics(
        y_val,
        scores,
        fraction=0.01
    )

    # --------------------------------------------------------
    # CONNECTIVITY STATISTICS
    # --------------------------------------------------------

    # How many different cards use this entity?
    card_count = (
        tmp.groupby(feature)["card1"]
        .nunique()
    )

    # Number of entities shared by >=2 cards
    multi_card_fraction = (
        (card_count >= 2).mean()
    )

    # Number shared by >=5 cards
    five_plus_fraction = (
        (card_count >= 5).mean()
    )

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    results.append({
        "feature": feature,

        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,

        "Top1_precision": precision_1,
        "Top1_lift": lift_1,

        "unique_entities": tmp[feature].nunique(),

        "multi_card_fraction": multi_card_fraction,
        "5plus_card_fraction": five_plus_fraction,

        "median_cards_per_entity":
            card_count.median(),

        "max_cards_per_entity":
            card_count.max()
    })


# ============================================================
# RESULTS
# ============================================================

entity_results = (
    pd.DataFrame(results)
    .sort_values("PR_AUC", ascending=False)
    .reset_index(drop=True)
)

pd.set_option("display.max_columns", None)

print("\n================ ENTITY SCREENING ================\n")

print(
    entity_results.round(4).to_string(index=False)
)

Testing: DeviceInfo
Testing: addr1
Testing: id_19
Testing: id_20
Testing: id_31
Testing: id_06
Testing: id_30
Testing: id_33
Testing: id_05
Testing: id_08
Testing: id_17
Testing: id_21
Testing: addr2
Testing: card2
Testing: card3
Testing: card5

================ ENTITY SCREENING ================

   feature  PR_AUC  ROC_AUC  Top1_precision  Top1_lift  unique_entities  multi_card_fraction  5plus_card_fraction  median_cards_per_entity  max_cards_per_entity
DeviceInfo  0.1370   0.6388          0.3903    11.1549             1686               0.6851               0.3458                      3.0                  4391
     card2  0.1262   0.7418          0.2540     7.2591              500               0.8800               0.4120                      4.0                  5052
     id_19  0.1111   0.6743          0.2667     7.6221              517               0.8781               0.6596                      7.0                  2572
     id_20  0.1058   0.6758          0.2540     7.2591    

In [28]:
print("TransactionDT" in train_transaction.columns)

if "TransactionDT" in train_transaction.columns:
    print(train_transaction["TransactionDT"].describe())

True
count    5.905400e+05
mean     7.372311e+06
std      4.617224e+06
min      8.640000e+04
25%      3.027058e+06
50%      7.306528e+06
75%      1.124662e+07
max      1.581113e+07
Name: TransactionDT, dtype: float64


In [29]:

SMOOTHING = 20
GLOBAL_FRAUD_RATE = train_df["isFraud"].mean()


def learn_entity_risk(train_data, feature):
    
    tmp = train_data[
        ["card1", feature, "isFraud"]
    ].dropna(subset=[feature])

    tx_count = (
        tmp.groupby(feature)
        .size()
    )

    fraud_count = (
        tmp.groupby(feature)["isFraud"]
        .sum()
    )

    # Smoothed fraud probability
    risk = (
        fraud_count + SMOOTHING * GLOBAL_FRAUD_RATE
    ) / (
        tx_count + SMOOTHING
    )

    return risk

In [30]:
def evaluate_entity_set(
    train_data,
    val_data,
    features
):

    risk_scores = []

    for feature in features:

        entity_risk = learn_entity_risk(
            train_data,
            feature
        )

        score = (
            val_data[feature]
            .map(entity_risk)
            .fillna(GLOBAL_FRAUD_RATE)
            .values
        )

        risk_scores.append(score)

    risk_matrix = np.column_stack(risk_scores)

    # -------------------------------------------------
    # Two complementary views
    # -------------------------------------------------

    max_risk = risk_matrix.max(axis=1)

    mean_risk = risk_matrix.mean(axis=1)

    # Combined score
    combined_score = (
        0.7 * max_risk +
        0.3 * mean_risk
    )

    y = val_data["isFraud"].values

    pr_auc = average_precision_score(
        y,
        combined_score
    )

    roc_auc = roc_auc_score(
        y,
        combined_score
    )

    # Top 1%
    k = max(1, int(len(y) * 0.01))

    order = np.argsort(-combined_score)

    top_labels = y[order[:k]]

    precision = top_labels.mean()

    lift = precision / y.mean()

    return {
        "features": features,
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "Top1_precision": precision,
        "Top1_lift": lift
    }

In [31]:
selected = ["DeviceInfo"]

remaining = [
    f for f in candidate_features
    if f not in selected
]

history = []

# Baseline
baseline = evaluate_entity_set(
    train_df,
    val_df,
    selected
)

history.append(baseline)

print("\nSTART")
print(baseline)


# =====================================================
# GREEDY FORWARD SELECTION
# =====================================================

while len(remaining) > 0:

    candidates = []

    print(
        f"\nTesting additions to: {selected}"
    )

    for feature in remaining:

        trial_features = selected + [feature]

        result = evaluate_entity_set(
            train_df,
            val_df,
            trial_features
        )

        candidates.append(result)

        print(
            f"+ {feature:<12} "
            f"PR-AUC={result['PR_AUC']:.4f} "
            f"Lift={result['Top1_lift']:.2f}"
        )

    # Best addition
    best = max(
        candidates,
        key=lambda x: x["PR_AUC"]
    )

    previous_pr = history[-1]["PR_AUC"]

    gain = (
        best["PR_AUC"] -
        previous_pr
    )

    print("\nBEST ADDITION:")
    print(best["features"])

    print(
        f"PR-AUC gain = {gain:.5f}"
    )

    # -------------------------------------------------
    # STOPPING CONDITION
    # -------------------------------------------------

    if gain < 0.002:

        print(
            "\nStopping: incremental gain too small."
        )

        break

    # Accept feature
    selected = best["features"]

    history.append(best)

    remaining.remove(
        selected[-1]
    )


print("\n================================")
print("FINAL GREEDY SELECTION")
print("================================")

for i, result in enumerate(history):

    print(
        f"Step {i}: "
        f"{result['features']} | "
        f"PR-AUC={result['PR_AUC']:.4f} | "
        f"Lift={result['Top1_lift']:.2f}"
    )


START
{'features': ['DeviceInfo'], 'PR_AUC': 0.1369514286456593, 'ROC_AUC': 0.6388104284460954, 'Top1_precision': np.float64(0.3903471634208298), 'Top1_lift': np.float64(11.15488090426019)}

Testing additions to: ['DeviceInfo']
+ addr1        PR-AUC=0.1625 Lift=11.01
+ id_19        PR-AUC=0.1742 Lift=11.45
+ id_20        PR-AUC=0.1751 Lift=11.86
+ id_31        PR-AUC=0.1572 Lift=11.49
+ id_06        PR-AUC=0.1533 Lift=10.99
+ id_30        PR-AUC=0.1467 Lift=11.15
+ id_33        PR-AUC=0.1452 Lift=11.32
+ id_05        PR-AUC=0.1545 Lift=10.74
+ id_08        PR-AUC=0.1383 Lift=11.13
+ id_17        PR-AUC=0.1604 Lift=11.30
+ id_21        PR-AUC=0.1428 Lift=11.69
+ addr2        PR-AUC=0.1532 Lift=11.32
+ card2        PR-AUC=0.1983 Lift=12.29
+ card3        PR-AUC=0.1600 Lift=11.15
+ card5        PR-AUC=0.1592 Lift=11.66

BEST ADDITION:
['DeviceInfo', 'card2']
PR-AUC gain = 0.06130

Testing additions to: ['DeviceInfo', 'card2']
+ addr1        PR-AUC=0.1995 Lift=12.15
+ id_19        PR-AUC=

In [32]:
results = []

for feature in candidate_features:

    # Ignore missing values
    temp = df[[feature, "card1", "isFraud"]].dropna(
        subset=[feature, "card1"]
    )

    # Each entity value -> cards using it
    entity_stats = (
        temp.groupby(feature)
        .agg(
            transactions=("card1", "size"),
            unique_cards=("card1", "nunique"),
            frauds=("isFraud", "sum")
        )
    )

    # How many cards does each entity connect?
    cards_per_entity = entity_stats["unique_cards"]

    n_entities = len(entity_stats)

    # --------------------------------------------------------
    # Entity size statistics
    # --------------------------------------------------------

    multi_card = (cards_per_entity >= 2).sum()
    five_plus = (cards_per_entity >= 5).sum()
    ten_plus = (cards_per_entity >= 10).sum()
    fifty_plus = (cards_per_entity >= 50).sum()
    hundred_plus = (cards_per_entity >= 100).sum()

    # --------------------------------------------------------
    # Hub dominance
    #
    # What fraction of all card-entity memberships comes
    # from very large entities?
    # --------------------------------------------------------

    total_card_memberships = cards_per_entity.sum()

    if total_card_memberships > 0:

        pct_from_50plus = (
            cards_per_entity[cards_per_entity >= 50].sum()
            / total_card_memberships
        )

        pct_from_100plus = (
            cards_per_entity[cards_per_entity >= 100].sum()
            / total_card_memberships
        )

    else:
        pct_from_50plus = 0
        pct_from_100plus = 0

    # --------------------------------------------------------
    # Fraud signal of entities
    # --------------------------------------------------------

    entity_fraud_rate = (
        entity_stats["frauds"] /
        entity_stats["transactions"]
    )

    # Only look at entities that have at least 2 cards
    meaningful_fraud_rates = entity_fraud_rate[
        cards_per_entity >= 2
    ]

    if len(meaningful_fraud_rates) > 0:
        median_fraud_rate = meaningful_fraud_rates.median()
        p90_fraud_rate = meaningful_fraud_rates.quantile(0.90)

        high_risk_entities = (
            meaningful_fraud_rates >= 0.10
        ).mean()
    else:
        median_fraud_rate = 0
        p90_fraud_rate = 0
        high_risk_entities = 0

    # --------------------------------------------------------
    # Card-sharing potential
    #
    # An entity connecting k cards can potentially create
    # k*(k-1)/2 card-card relationships.
    #
    # We calculate this WITHOUT actually generating the edges.
    # --------------------------------------------------------

    potential_pairs = (
        cards_per_entity * (cards_per_entity - 1) / 2
    ).sum()

    # Cap display because gigantic hubs can produce huge numbers
    # but we keep the real value internally.
    
    results.append({
        "feature": feature,
        "unique_entities": n_entities,

        "multi_card_entities": multi_card,
        "5plus_card_entities": five_plus,
        "10plus_card_entities": ten_plus,
        "50plus_card_entities": fifty_plus,
        "100plus_card_entities": hundred_plus,

        "median_cards_per_entity": cards_per_entity.median(),
        "p90_cards_per_entity": cards_per_entity.quantile(0.90),
        "max_cards_per_entity": cards_per_entity.max(),

        "pct_memberships_from_50plus":
            pct_from_50plus,

        "pct_memberships_from_100plus":
            pct_from_100plus,

        "median_entity_fraud_rate":
            median_fraud_rate,

        "p90_entity_fraud_rate":
            p90_fraud_rate,

        "high_risk_entity_fraction":
            high_risk_entities,

        "potential_card_pairs":
            potential_pairs
    })


graph_quality = pd.DataFrame(results)

# ------------------------------------------------------------
# Sort by useful multi-card entities
# ------------------------------------------------------------

graph_quality = graph_quality.sort_values(
    "5plus_card_entities",
    ascending=False
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

print("\n================ GRAPH ENTITY QUALITY ================\n")

print(graph_quality.to_string(index=False))


================ GRAPH ENTITY QUALITY ================

   feature  unique_entities  multi_card_entities  5plus_card_entities  10plus_card_entities  50plus_card_entities  100plus_card_entities  median_cards_per_entity  p90_cards_per_entity  max_cards_per_entity  pct_memberships_from_50plus  pct_memberships_from_100plus  median_entity_fraud_rate  p90_entity_fraud_rate  high_risk_entity_fraction  potential_card_pairs
DeviceInfo             1786                 1237                  659                   352                    31                     15                      3.0                  19.0                  4846                     0.596878                      0.556903                  0.000000               0.381522                   0.240905            21461607.0
     id_19              522                  469                  368                   243                    75                     49                      9.0                  86.9                  2898            

In [34]:
# ============================================================
# TOP ENTITY VALUES BY NUMBER OF CONNECTED CARDS
# ============================================================

for feature in candidate_features:

    temp = df[
        [feature, "card1", "isFraud"]
    ].dropna(subset=[feature, "card1"])

    stats = (
        temp.groupby(feature)
        .agg(
            transactions=("card1", "size"),
            unique_cards=("card1", "nunique"),
            frauds=("isFraud", "sum")
        )
    )

    stats["fraud_rate"] = (
        stats["frauds"] / stats["transactions"]
    )

    stats = stats.sort_values(
        "unique_cards",
        ascending=False
    )

    print("\n" + "=" * 80)
    print(f"TOP HUBS: {feature}")
    print("=" * 80)

    print(
        stats.head(10).to_string()
    )


TOP HUBS: DeviceInfo
                       transactions  unique_cards  frauds  fraud_rate
DeviceInfo                                                           
Windows                       47722          4846    3121    0.065400
iOS Device                    19782          3104    1240    0.062683
MacOS                         12573          2287     278    0.022111
Trident/7.0                    7440          1868      96    0.012903
rv:11.0                        1901           702      76    0.039979
rv:57.0                         962           422     103    0.107069
SM-G930V Build/NRD90M           274           167       0    0.000000
SM-G950U Build/NRD90M           290           164      16    0.055172
SM-G955U Build/NRD90M           328           163      33    0.100610
SAMSUNG                         235           157       5    0.021277

TOP HUBS: addr1
       transactions  unique_cards  frauds  fraud_rate
addr1                                                
299.0        